# AWS Data Query Notebook

Use this notebook to query Redshift and S3 directly.

## One-time setup (per session)

Run this in a terminal from project root (keep that terminal open):
`python3 scripts/connect_redshift.py --env data_analyst_dev`

Then come back to this notebook and run the Redshift cells.

You can change `--env` to: `data_analyst_dev`, `data_qa_dev`, `data_analyst_preprod`.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

project_root = Path.cwd()
if not (project_root / 'src').exists():
    project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


def _df_display(self, rows=100):
    display(self.head(rows))


if not hasattr(pd.DataFrame, 'display'):
    pd.DataFrame.display = _df_display

print('Project root:', project_root)
print('DataFrame display helper ready: use df.display()')

Project root: /Users/kkrishna/Downloads/pp/IB_Prjs/1IB_AWS_DataQuery_xtension
DataFrame display helper ready: use df.display()


In [2]:
import os
import socket

from src.config import get_redshift_config, get_selected_redshift_env, list_configured_redshift_envs
from src.redshift_client import run_redshift_query


def _port_open(host: str, port: int, timeout: float = 1.0) -> bool:
    try:
        with socket.create_connection((host, int(port)), timeout=timeout):
            return True
    except OSError:
        return False


def _normalize_env_name(env_name: str) -> str:
    return env_name.strip().upper().replace("-", "_")


def detect_active_tunnel_envs(env_names: list[str]) -> list[str]:
    active = []
    for env in env_names:
        try:
            cfg = get_redshift_config(env)
        except Exception:
            continue
        host = str(cfg.get("host", "")).strip()
        port = int(cfg.get("port", 0))
        if host in {"localhost", "127.0.0.1"} and port > 0 and _port_open(host, port):
            active.append(env)
    return active


configured_envs = list_configured_redshift_envs()
default_env = get_selected_redshift_env()
active_tunnel_envs = detect_active_tunnel_envs(configured_envs)

# Options:
# - "auto" -> use detected tunnel env(s), else fallback to default env from .env
# - "all" -> run on all configured envs
# - "data_analyst_preprod" -> run on one env
# - ["data_analyst_dev", "data_analyst_preprod"] -> run on multiple envs
target_envs = "auto"

if target_envs == "auto":
    selected_envs = active_tunnel_envs if active_tunnel_envs else [default_env]
elif target_envs == "all":
    selected_envs = configured_envs
elif isinstance(target_envs, str):
    selected_envs = [target_envs]
else:
    selected_envs = list(target_envs)

# Some clusters do not have public schema. Override preprod schema for this session.
session_schema_overrides = {
    "data_analyst_preprod": "edw_asis",
    "preprod": "edw_asis",
}
for env in selected_envs:
    schema = session_schema_overrides.get(env)
    if not schema:
        continue
    normalized = _normalize_env_name(env)
    os.environ[f"REDSHIFT_SCHEMA_{normalized}"] = schema
    os.environ[f"{normalized}_SCHEMA"] = schema

current_env = selected_envs[0]

print("Configured envs     :", configured_envs)
print("Active tunnel envs  :", active_tunnel_envs)
print("Default env in .env :", default_env)
print("Selected env(s)     :", selected_envs)
print("Schema overrides    :", {k: v for k, v in session_schema_overrides.items() if k in selected_envs})

Configured envs     : ['data_analyst_dev', 'data_analyst_preprod', 'data_qa_dev', 'preprod']
Active tunnel envs  : ['data_analyst_preprod', 'preprod']
Default env in .env : data_analyst_dev
Selected env(s)     : ['data_analyst_preprod', 'preprod']
Schema overrides    : {'data_analyst_preprod': 'edw_asis', 'preprod': 'edw_asis'}


In [3]:
# Connection test cell: run this before your main query cell.
test_sql = "select current_user as user_name, current_database() as db_name, current_date as run_date"

status_rows = []
for env in selected_envs:
    cfg = get_redshift_config(env)
    print("\n---", env, "---")
    print('Host / Port  :', f"{cfg['host']}:{cfg['port']}")
    print('Database/User:', f"{cfg['database']} / {cfg['user']}")
    print('Password set :', bool(cfg.get('password')))
    try:
        df_conn_test = run_redshift_query(test_sql, env_name=env)
        print('Connection test: SUCCESS')
        status_rows.append({"env": env, "status": "SUCCESS", "error": ""})
    except Exception as exc:
        print('Connection test: FAILED')
        print(type(exc).__name__, str(exc))
        status_rows.append({"env": env, "status": "FAILED", "error": f"{type(exc).__name__}: {exc}"})

df_conn_status = pd.DataFrame(status_rows)
df_conn_status.display()


--- data_analyst_preprod ---
Host / Port  : localhost:54392
Database/User: ib-dl-it / kkrishna
Password set : True
Connection test: SUCCESS

--- preprod ---
Host / Port  : localhost:54392
Database/User: ib-dl-it / kkrishna
Password set : True
Connection test: SUCCESS


,env,status,error
0,data_analyst_preprod,SUCCESS,
1,preprod,SUCCESS,


In [4]:
# Write your Redshift query below and run this cell.
redshift_sql = """
select table_schema, table_name, column_name from information_schema.columns
where table_schema = 'edw_asis'
and table_name ilike '%oracle_fusion%'
order by table_name
"""

if len(selected_envs) == 1:
    df_redshift = run_redshift_query(redshift_sql, env_name=selected_envs[0])
    df_redshift.display()
else:
    frames = []
    errors = []
    for env in selected_envs:
        try:
            df_env = run_redshift_query(redshift_sql, env_name=env).copy()
            df_env.insert(0, "_env", env)
            frames.append(df_env)
        except Exception as exc:
            errors.append(f"{env}: {type(exc).__name__}: {exc}")

    if frames:
        df_redshift = pd.concat(frames, ignore_index=True)
        df_redshift.display()
    else:
        raise RuntimeError("Query failed for all selected envs. Errors: " + " | ".join(errors))

,_env,table_schema,table_name,column_name
0,data_analyst_preprod,edw_asis,oracle_fusion_dootop_fulfill_line,row_id
1,data_analyst_preprod,edw_asis,oracle_fusion_dootop_fulfill_line,document_references_doc_id
2,data_analyst_preprod,edw_asis,oracle_fusion_dootop_fulfill_line,edw_act_fl
3,data_analyst_preprod,edw_asis,oracle_fusion_dootop_fulfill_line,edw_user_nm
4,data_analyst_preprod,edw_asis,oracle_fusion_dootop_fulfill_line,edw_src_cd
...,...,...,...,...
95,data_analyst_preprod,edw_asis,oracle_fusion_dootop_fulfill_line,document_references_doc_subline_id
96,data_analyst_preprod,edw_asis,oracle_fusion_dootop_fulfill_line,document_references_doc_subline_context_id
97,data_analyst_preprod,edw_asis,oracle_fusion_dootop_fulfill_line,document_references_doc_subline_alt_user_key
98,data_analyst_preprod,edw_asis,oracle_fusion_dootop_fulfill_line,document_references_doc_ref_type


In [5]:
from src.s3_client import read_s3_file, query_s3_path

# Set a real S3 path before running these cells.
s3_path = "s3://your-bucket/path/file.parquet"
print('S3 path:', s3_path)

S3 path: s3://your-bucket/path/file.parquet


In [6]:
df_s3_preview = read_s3_file(s3_path, limit=20)
df_s3_preview.display()

AccessDenied: An error occurred (AccessDenied) when calling the GetObject operation: Access Denied

In [ ]:
s3_sql = f"select * from read_parquet('{s3_path}') limit 20"
df_s3_sql = query_s3_path(s3_path, sql=s3_sql)
df_s3_sql.display()